# 📓 04. Fair-Value Pricing, Anomaly Detection & Case Study Resolution
> **Đề tài:** Phân tích Không gian - Kinh tế thị trường phòng trọ TP.HCM & Bình Dương (IE313)  

---
## 🎯 Mục tiêu Notebook:
1. Huấn luyện và so sánh các mô hình hồi quy định giá (**Ridge, Random Forest, LightGBM**) dựa trên $R^2$, MAE, RMSE, MAPE.
2. Phân tích tầm quan trọng đặc trưng (**Feature Importance**) giải thích cơ chế định giá.
3. Huấn luyện mô hình phát hiện bất thường (**Isolation Forest**) trên phần dư định giá (**Valuation Gap %**).
4. **Giải quyết Case Study thực tế:** Tách biệt và phân loại danh sách các tin đăng **Kèo thơm giá hời (Good Deals)** vs **Bẫy tin ảo câu cọc (Scam Risks)**.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path('..').resolve()
sys.path.append(str(BASE_DIR))

from src.models import run_full_modeling_pipeline

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Huấn luyện & Đánh giá Benchmark Mô hình Định giá

In [ ]:
df_scored, summary = run_full_modeling_pipeline()

benchmark_df = pd.DataFrame(summary['benchmarks']).T
print('=== BẢNG SO SÁNH HIỆU NĂNG MÔ HÌNH HỒI QUY ===')
display(benchmark_df)

# Vẽ biểu đồ so sánh R2 và MAE
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
benchmark_df['R2'].plot(kind='bar', ax=ax[0], color='#2b5c8f', edgecolor='black')
ax[0].set_title('Hệ số Xác định (R² Score) - Càng cao càng tốt', fontweight='bold')
ax[0].set_ylabel('R²')

benchmark_df['MAE_Million'].plot(kind='bar', ax=ax[1], color='#e24a33', edgecolor='black')
ax[1].set_title('Sai số Tuyệt đối Trung bình (MAE, Triệu VNĐ) - Càng thấp càng tốt', fontweight='bold')
ax[1].set_ylabel('Triệu VNĐ')

plt.tight_layout()
plt.show()

## 2. Giải quyết Case Study: Radar Phát hiện Kèo thơm vs Bẫy Tin ảo

In [ ]:
print('=== PHÂN BỔ TRẠNG THÁI QUYẾT ĐỊNH (DEAL STATUS) ===')
display(df_scored['deal_status'].value_counts())

# Đồ thị Valuation Gap
plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_scored.sample(min(800, len(df_scored)), random_state=42),
    x='predicted_price',
    y='price_million',
    hue='deal_status',
    palette={'GOOD_DEAL': '#2ca02c', 'FAIR_PRICE': '#1f77b4', 'OVERPRICED': '#ff7f0e', 'SCAM_RISK': '#d62728', 'SHARED_DORM': '#9e9e9e'},
    alpha=0.7,
)
max_val = max(df_scored['predicted_price'].max(), df_scored['price_million'].max())
plt.plot([0, max_val], [0, max_val], 'k--', label='Đường Cân bằng Giá (Fair-Value Line)')
plt.title('Đồ thị Phân loại Tin đăng: Giá Thực tế vs. Giá Hợp lý AI Dự đoán', fontsize=12, fontweight='bold')
plt.xlabel('Giá Hợp lý do Mô hình AI Dự đoán (Triệu VNĐ/tháng)')
plt.ylabel('Giá Thực tế Đăng trên Chợ Tốt (Triệu VNĐ/tháng)')
plt.legend()
plt.tight_layout()
plt.show()

print('\n--- TOP 5 KÈO THƠM GIÁ HỜI (GOOD DEALS) TIÊU BIỂU ---')
good_deals = df_scored[df_scored['deal_status'] == 'GOOD_DEAL'].sort_values('valuation_gap_pct', ascending=True)
display(good_deals[['ad_id', 'area_name', 'size_m2', 'price_million', 'predicted_price', 'valuation_gap_pct', 'seller_type']].head(5))

print('\n--- TOP 5 CẢNH BÁO BẪY TIN ẢO CÂU CỌC (SCAM RISKS) ---')
scam_risks = df_scored[df_scored['deal_status'] == 'SCAM_RISK'].sort_values('valuation_gap_pct', ascending=True)
display(scam_risks[['ad_id', 'area_name', 'size_m2', 'price_million', 'predicted_price', 'valuation_gap_pct', 'seller_type']].head(5))